In [1]:
import pandas as pd
import numpy as np

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [2]:
train_df = pd.read_csv(
    "../data/processed/train_with_rul.csv"
)

train_df.head()

,unit_number,time_cycle,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,max_cycle,RUL
0,1,1,42.0049,0.8400,100.0,445.00,549.68,1343.43,1112.93,3.91,...,8074.83,9.3335,0.02,330,2212,100.00,10.62,6.3670,321,320
1,1,2,20.0020,0.7002,100.0,491.19,606.07,1477.61,1237.50,9.35,...,8046.13,9.1913,0.02,361,2324,100.00,24.37,14.6552,321,319
2,1,3,42.0038,0.8409,100.0,445.00,548.95,1343.12,1117.05,3.91,...,8066.62,9.4007,0.02,329,2212,100.00,10.48,6.4213,321,318
3,1,4,42.0000,0.8400,100.0,445.00,548.70,1341.24,1118.03,3.91,...,8076.05,9.3369,0.02,328,2212,100.00,10.54,6.4176,321,317
4,1,5,25.0063,0.6207,60.0,462.54,536.10,1255.23,1033.59,7.05,...,7865.80,10.8366,0.02,305,1915,84.93,14.03,8.6754,321,316


In [3]:
DROP_SENSORS = [
    "sensor_10",
    "sensor_15",
    "sensor_16"
]

train_df = train_df.drop(
    columns=DROP_SENSORS
)

print(train_df.shape)

(61249, 25)


In [4]:
train_df["engine_age"] = (
    train_df["time_cycle"]
)

train_df["life_fraction"] = (
    train_df["time_cycle"]/train_df["max_cycle"]
)

In [5]:
op_cols = [
    "op_setting_1",
    "op_setting_2",
    "op_setting_3"
]

kmeans = KMeans(
    n_clusters=6,
    random_state=42,
    n_init=10
)

train_df["operating_cluster"] = (
    kmeans.fit_predict(
        train_df[op_cols]
    )
)

In [6]:
train_df["operating_cluster"].value_counts()

operating_cluster
0    15395
4     9238
1     9224
5     9162
2     9139
3     9091
Name: count, dtype: int64

In [7]:
sensor_cols = [
    col
    for col in train_df.columns
    if "sensor_" in col
]

In [8]:
for sensor in sensor_cols:

    train_df[
        f"{sensor}_norm"
    ] = (
        train_df.groupby(
            "operating_cluster"
        )[sensor]
        .transform(
            lambda x:(x - x.mean())/x.std()
        )
    )

In [9]:
TOP_SENSORS = [
    "sensor_11",
    "sensor_14",
    "sensor_4",
    "sensor_17"
]

In [10]:
for sensor in TOP_SENSORS:

    train_df[
        f"{sensor}_rolling_mean"
    ] = (
        train_df
        .groupby("unit_number")[sensor]
        .transform(
            lambda x:
            x.rolling(
                window=5,
                min_periods=1
            ).mean()
        )
    )

In [11]:
for sensor in TOP_SENSORS:

    train_df[
        f"{sensor}_delta"
    ] = (
        train_df
        .groupby("unit_number")[sensor]
        .diff()
    )

In [12]:
train_df.fillna(
    0,
    inplace=True
)

,unit_number,time_cycle,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_20_norm,sensor_21_norm,sensor_11_rolling_mean,sensor_14_rolling_mean,sensor_4_rolling_mean,sensor_17_rolling_mean,sensor_11_delta,sensor_14_delta,sensor_4_delta,sensor_17_delta
0,1,1,42.0049,0.8400,100.0,445.00,549.68,1343.43,1112.93,3.91,...,-0.024002,-0.079912,41.6900,8074.830000,1112.930000,330.0,0.00,0.00,0.00,0.0
1,1,2,20.0020,0.7002,100.0,491.19,606.07,1477.61,1237.50,9.35,...,-0.940905,-0.634450,42.8150,8060.480000,1175.215000,345.5,2.25,-28.70,124.57,31.0
2,1,3,42.0038,0.8409,100.0,445.00,548.95,1343.12,1117.05,3.91,...,-1.204026,0.681673,42.4300,8062.526667,1155.826667,340.0,-2.28,20.49,-120.45,-32.0
3,1,4,42.0000,0.8400,100.0,445.00,548.70,1341.24,1118.03,3.91,...,-0.698302,0.629779,42.2425,8065.907500,1146.377500,337.0,0.02,9.43,0.98,-1.0
4,1,5,25.0063,0.6207,60.0,462.54,536.10,1255.23,1033.59,7.05,...,-2.185794,1.153795,41.0900,8025.886000,1123.820000,330.6,-5.20,-210.25,-84.44,-23.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61244,249,251,9.9998,0.2500,100.0,489.05,605.33,1516.36,1315.28,10.52,...,2.463164,2.965277,42.6580,8094.416000,1183.226000,342.8,8.87,276.08,257.88,65.0
61245,249,252,0.0028,0.0015,100.0,518.67,643.42,1598.92,1426.77,14.62,...,1.588919,2.142538,43.7440,8104.762000,1240.596000,355.2,1.96,-0.22,111.49,24.0
61246,249,253,0.0029,0.0000,100.0,518.67,643.68,1607.72,1430.56,14.62,...,3.164683,2.877473,44.4260,8122.230000,1273.668000,360.6,0.20,8.47,3.79,-1.0
61247,249,254,35.0046,0.8400,100.0,449.44,555.77,1381.29,1148.18,5.48,...,2.804010,1.833043,44.4200,8120.070000,1275.638000,361.4,-5.71,-68.30,-282.38,-58.0


In [13]:
train_df.to_csv(
    "../data/processed/train_engineered.csv",
    index=False
)

print(
    "Feature engineering complete."
)

Feature engineering complete.


In [14]:
train_df.shape

(61249, 54)

In [15]:
train_df.columns.tolist()[-20:]

['sensor_7_norm',
 'sensor_8_norm',
 'sensor_9_norm',
 'sensor_11_norm',
 'sensor_12_norm',
 'sensor_13_norm',
 'sensor_14_norm',
 'sensor_17_norm',
 'sensor_18_norm',
 'sensor_19_norm',
 'sensor_20_norm',
 'sensor_21_norm',
 'sensor_11_rolling_mean',
 'sensor_14_rolling_mean',
 'sensor_4_rolling_mean',
 'sensor_17_rolling_mean',
 'sensor_11_delta',
 'sensor_14_delta',
 'sensor_4_delta',
 'sensor_17_delta']